# Data Transformation

## Install Dependencies

In [1]:
!./install_packages.sh

## Imports and Environment Setup

In [5]:
import sys
import io
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import json
import geojson
from datetime import datetime
import time
from pathlib import Path
import shapely
from shapely.geometry import box, Point, Polygon
import matplotlib.pyplot as plt
from pyproj import Transformer
import zipfile
import subprocess
from io import StringIO
import contextily as ctx
from osgeo import gdal, osr

# Enable GDAL exceptions
gdal.UseExceptions()
osr.UseExceptions()

# Import source code
sys.path.insert(0, 'farsite')
from farsite import *

## Define Ignition Constraints

In [6]:
IGNITION_LAT = 38.9014
IGNITION_LON = -120.0306
FIREMAP_USERNAME = 'YOUR_USERNAME'  ## Change to your Firemap account username
FIREMAP_PASSWORD = 'YOUR_PASSWORD'  ## Change to your Firemap account password

## Compile Landscape Data
Use ``lcpmake`` executable to compile .tif files into ``.lcp`` format file for FARSITE input.

In [ ]:
# Define landscape generation function
def generate_lcp_from_rasters(
    output_path,
    elevation_asc,
    slope_asc,
    aspect_asc,
    fuel_asc,
    canopy_cover_asc,
    canopy_height_asc,
    canopy_base_asc,
    canopy_density_asc,
    latitude=None,
    fuel_model="fb40",
    verbose=True
):
    """
    Generate FARSITE landscape (.lcp) file from ASCII rasters using lcpmake.
    
    Args:
        output_path: Output .lcp file path
        elevation_asc: Elevation raster (.asc) in meters
        slope_asc: Slope raster (.asc) in percent
        aspect_asc: Aspect raster (.asc) in degrees (0-360)
        fuel_asc: Fuel model raster (.asc) - integers matching fuel_model
        canopy_cover_asc: Canopy cover (.asc) in percent (0-100)
        canopy_height_asc: Canopy height (.asc) in meters * 10
        canopy_base_asc: Canopy base height (.asc) in meters * 10
        canopy_density_asc: Canopy bulk density (.asc) in kg/m³ * 100
        latitude: Center latitude in decimal degrees (auto-detected if None)
        fuel_model: Fuel model type - "fb40" (FBFM40) or "fb13" (FBFM13)
        verbose: Print lcpmake command
        
    Returns:
        Path to generated .lcp file
    """
    output_path = Path(output_path)
    lcpmake_exe = Path(LCPMAKE_EXECUTABLE)
    
    if not lcpmake_exe.exists():
        raise FileNotFoundError(
            f"lcpmake executable not found at {lcpmake_exe}\n"
            # f"Place lcpmake in {SCRIPT_DIR}/"
        )
    
    # Auto-detect latitude from elevation raster if not provided
    if latitude is None:
        ds = gdal.Open(str(elevation_asc))
        if ds:
            gt = ds.GetGeoTransform()
            proj = ds.GetProjection()
            x_center = gt[0] + (ds.RasterXSize / 2) * gt[1]
            y_center = gt[3] + (ds.RasterYSize / 2) * gt[5]
            
            src_srs = osr.SpatialReference()
            src_srs.ImportFromWkt(proj)
            dst_srs = osr.SpatialReference()
            dst_srs.ImportFromEPSG(4326)
            transform = osr.CoordinateTransformation(src_srs, dst_srs)
            lon, lat, _ = transform.TransformPoint(x_center, y_center)
            latitude = lat
            ds = None
            if verbose:
                print(f"Auto-detected latitude: {latitude:.4f}")
    
    # Build lcpmake command
    cmd = [
        str(lcpmake_exe),
        "-latitude", str(latitude),
        "-landscape", str(output_path.with_suffix('')),
        "-elevation", str(elevation_asc),
        "-slope", str(slope_asc),
        "-aspect", str(aspect_asc),
        "-fuel", str(fuel_asc),
        "-cover", str(canopy_cover_asc),
        "-height", str(canopy_height_asc),
        "-base", str(canopy_base_asc),
        "-density", str(canopy_density_asc),
    ]
    
    if fuel_model.lower() in ["fb40", "fbfm40", "40"]:
        cmd.append("-fb40")
    elif fuel_model.lower() in ["fb13", "fbfm13", "13"]:
        cmd.append("-fb13")
    else:
        raise ValueError(f"Unknown fuel model: {fuel_model}")
    
    if verbose:
        print("\nRunning lcpmake command:")
        print(" ".join(cmd))
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        raise RuntimeError(
            f"lcpmake failed with return code {result.returncode}\n"
            f"stdout: {result.stdout}\nstderr: {result.stderr}"
        )
    
    final_path = output_path.with_suffix('.lcp')
    
    if not final_path.exists():
        raise RuntimeError(f"lcpmake succeeded but output file not found: {final_path}")
    
    if verbose:
        print(f"\n✓ Landscape file created: {final_path}")
        print(f"  Size: {final_path.stat().st_size / 1024:.1f} KB")
    
    return final_path

In [ ]:
lcp_path_returned = generate_lcp_from_rasters(
                    output_path=LCP_PATH,
                    elevation_asc=landfire_data['elevation'],
                    slope_asc=landfire_data['slope'],
                    aspect_asc=landfire_data['aspect'],
                    fuel_asc=landfire_data['fuel'],
                    canopy_cover_asc=landfire_data['canopy_cover'],
                    canopy_height_asc=landfire_data['canopy_height'],
                    canopy_base_asc=landfire_data['canopy_base'],
                    canopy_density_asc=landfire_data['canopy_density'],
                    latitude=IGNITION_LAT
)